# MuJoCo vs Pinocchio — `Pinocchio_Wrapper`의 모든 양을 순수 MuJoCo로 재현 가능한가?

G1 floating-base 모델에서 동일 상태 `(q, dq)`를 두 라이브러리에 넣고, 각 양을 수치적으로 비교한다.

## 변환 규약
| | MuJoCo | Pinocchio (FreeFlyer) |
|---|---|---|
| `qpos[3:7]` quaternion | `(w,x,y,z)` | `(x,y,z,w)` |
| free joint `qvel[0:3]` (linear) | **world frame** | **body local** |
| free joint `qvel[3:6]` (angular) | body local | body local |

→ Jacobian/M/nle 비교 시 좌표 변환 행렬 `T = blkdiag(R_body_to_world, I_3, I_{nv-6})` 필수.
  - velocity는 covariant: `qvel_mj = T @ dq_pin`
  - Jacobian: `J_pin = J_mj @ T`
  - mass matrix: `M_pin = T^T M_mj T`
  - generalized force: `f_pin = T^T f_mj`

In [ ]:
import sys
sys.path.insert(0, '/home/frlab/mj_opt/mj_sim/humanoid')

import numpy as np
import mujoco
import pinocchio as pin

from core import FloatingBaseRobotState, Pinocchio_Wrapper, Mujoco_Kernel
from core.pinocchio_wrapper import FOOT_SOLE_OFFSET

%load_ext autoreload
%autoreload 2

URDF = '/home/frlab/mj_opt/models/g1/urdf/g1_29dof.urdf'
PKG  = '/home/frlab/mj_opt/models/g1'
MJCF = '/home/frlab/mj_opt/models/scenes/g1/scene_29dof.xml'

PIN_TO_MJBODY = {
    'L_foot' : 'left_ankle_roll_link',
    'R_foot' : 'right_ankle_roll_link',
    'L_hand' : 'left_wrist_yaw_link',     # MJ 모델엔 rubber_hand 없음 → wrist_yaw로 대체
    'R_hand' : 'right_wrist_yaw_link',
    'base'   : 'pelvis',
}

## 1. 두 모델 인스턴스화 + 동일 상태 동기화

In [ ]:
wrapper = Pinocchio_Wrapper(URDF, PKG)
joint_names_pin = [wrapper.model.names[i] for i in range(2, wrapper.model.njoints)]
kernel = Mujoco_Kernel(MJCF, joint_names_pin_order=joint_names_pin)
kernel.reset_to_keyframe(0)

# 비대칭 상태 (yaw 회전 + 관절/속도 perturb)
rng = np.random.default_rng(0)
yaw0 = np.deg2rad(30.0)
kernel.data.qpos[3] = np.cos(yaw0/2)   # MJ quat w
kernel.data.qpos[6] = np.sin(yaw0/2)   # MJ quat z
kernel.data.qpos[7:] += 0.05 * rng.standard_normal(kernel.model.nq - 7)
kernel.data.qvel[:]  = 0.1 * rng.standard_normal(kernel.model.nv)
mujoco.mj_forward(kernel.model, kernel.data)

state = FloatingBaseRobotState()
kernel.update_robot_state(state)
wrapper.update_model(state.q, state.dq)

nv = wrapper.nv
R = wrapper.R_body_to_world
T = np.eye(nv); T[0:3, 0:3] = R

print(f'Pin mass = {wrapper.mass:.4f}')
print(f'MJ  mass = {sum(kernel.model.body_mass[1:]):.4f}')
print(f'nq, nv   = {kernel.model.nq}, {kernel.model.nv}')
print(f'\nMJ  qpos[:7] (w,x,y,z) = {kernel.data.qpos[:7]}')
print(f'Pin q[:7]    (x,y,z,w) = {state.q[:7]}')

## 2. 기구학 — 정확히 일치 ✅

`base_pos`, `com_pos`, `J_com`, frame `J_world`, frame placement 모두 **MuJoCo 단독으로 1e-5 이하 오차**로 재현 가능.

In [ ]:
# 2.1 base_pos
base_id = mujoco.mj_name2id(kernel.model, mujoco.mjtObj.mjOBJ_BODY, 'pelvis')
err_base = np.linalg.norm(wrapper.base_pos - kernel.data.xpos[base_id])
print(f'[base_pos]    err = {err_base:.3e}')

# 2.2 com_pos: subtree_com[1] = pelvis subtree → 로봇 전체 CoM
err_com = np.linalg.norm(wrapper.com_pos_world - kernel.data.subtree_com[1])
print(f'[com_pos]     err = {err_com:.3e}')

# 2.3 J_com (mj_jacSubtreeCom)
Jcom_mj = np.zeros((3, nv))
mujoco.mj_jacSubtreeCom(kernel.model, kernel.data, Jcom_mj, 1)
err_Jcom = np.linalg.norm(wrapper.J_com - Jcom_mj @ T)
print(f'[J_com]       err (Jcom_mj @ T) = {err_Jcom:.3e}')

In [ ]:
# 2.4 Frame Jacobian (foot sole, hand)
def mj_frame_jac_world(model, data, body_id, point_world):
    """point in world coords → 6×nv (linear stacked on angular). pin LOCAL_WORLD_ALIGNED 등가."""
    jacp = np.zeros((3, model.nv))
    jacr = np.zeros((3, model.nv))
    mujoco.mj_jac(model, data, jacp, jacr, point_world, body_id)
    return np.vstack([jacp, jacr])

for key in ['L_foot', 'R_foot', 'L_hand', 'R_hand']:
    body_name = PIN_TO_MJBODY[key]
    body_id   = mujoco.mj_name2id(kernel.model, mujoco.mjtObj.mjOBJ_BODY, body_name)
    if key in ('L_foot', 'R_foot'):
        R_body = kernel.data.xmat[body_id].reshape(3, 3)
        p_world = kernel.data.xpos[body_id] + R_body @ FOOT_SOLE_OFFSET
    else:
        p_world = kernel.data.xpos[body_id].copy()
    
    J_pin = wrapper.get_J_world(key)
    J_mj  = mj_frame_jac_world(kernel.model, kernel.data, body_id, p_world)
    err = np.linalg.norm(J_pin - J_mj @ T)
    note = '' if key in ('L_foot', 'R_foot') else '  ← MJ에선 wrist_yaw, Pin은 rubber_hand → 다름'
    print(f'[J_{key:6s}] err (J_mj @ T) = {err:.3e}{note}')

In [ ]:
# 2.5 Frame placement (foot sole)
for key in ['L_foot', 'R_foot']:
    body_id = mujoco.mj_name2id(kernel.model, mujoco.mjtObj.mjOBJ_BODY, PIN_TO_MJBODY[key])
    R_body  = kernel.data.xmat[body_id].reshape(3, 3)
    p_mj    = kernel.data.xpos[body_id] + R_body @ FOOT_SOLE_OFFSET
    p_pin   = wrapper.data.oMf[wrapper.fid[key]].translation
    print(f'[{key} sole]  err = {np.linalg.norm(p_pin - p_mj):.3e}')

# 2.6 Frame velocity (twist) — J_world @ qvel 로 재현
for key in ['L_foot', 'R_foot']:
    body_id = mujoco.mj_name2id(kernel.model, mujoco.mjtObj.mjOBJ_BODY, PIN_TO_MJBODY[key])
    R_body  = kernel.data.xmat[body_id].reshape(3, 3)
    p_world = kernel.data.xpos[body_id] + R_body @ FOOT_SOLE_OFFSET
    twist_mj = mj_frame_jac_world(kernel.model, kernel.data, body_id, p_world) @ kernel.data.qvel
    _, lin_pin, ang_pin = wrapper.get_ee_state_world(key)
    err_lin = np.linalg.norm(lin_pin - twist_mj[:3])
    err_ang = np.linalg.norm(ang_pin - twist_mj[3:])
    print(f'[{key} twist] lin err = {err_lin:.3e}, ang err = {err_ang:.3e}')

## 3. 동역학 — M, nle, g

두 모델은 같은 로봇이지만 URDF와 MJCF에서 **link inertia 정의가 약간 다를 수 있음** (소프트웨어 export 시 rounding, MuJoCo가 collision mesh로 재계산 등). Jacobian은 1e-5에서 일치하지만 mass matrix는 ~5e-2 정도 차이가 남는다.

In [ ]:
# 3.1 Mass matrix
M_pin = wrapper.M.copy()
M_mj  = np.zeros((nv, nv))
mujoco.mj_fullM(kernel.model, M_mj, kernel.data.qM)
M_mj_to_pin = T.T @ M_mj @ T

print(f'||M_pin - T^T M_mj T||_F = {np.linalg.norm(M_pin - M_mj_to_pin):.3e}')
print(f'||M_pin||_F              = {np.linalg.norm(M_pin):.3e}')
print(f'relative err = {np.linalg.norm(M_pin - M_mj_to_pin)/np.linalg.norm(M_pin)*100:.2f}%')

# 자기 자신 대칭성 확인
print(f'\nM_pin 대칭: ||M-M.T||={np.linalg.norm(M_pin - M_pin.T):.2e}')
print(f'M_mj  대칭: ||M-M.T||={np.linalg.norm(M_mj  - M_mj.T):.2e}')

# 3.2 nle = C·dq + g
nle_pin = wrapper.nle.copy()
nle_mj  = kernel.data.qfrc_bias.copy()
nle_mj_to_pin = T.T @ nle_mj
print(f'\n||nle_pin - T^T nle_mj|| = {np.linalg.norm(nle_pin - nle_mj_to_pin):.3e}')
print(f'||nle_pin||              = {np.linalg.norm(nle_pin):.3e}')

# 3.3 g (gravity) — qvel=0에서 qfrc_bias
qvel_save = kernel.data.qvel.copy()
kernel.data.qvel[:] = 0
mujoco.mj_forward(kernel.model, kernel.data)
g_mj = kernel.data.qfrc_bias.copy()
kernel.data.qvel[:] = qvel_save
mujoco.mj_forward(kernel.model, kernel.data)

g_pin = wrapper.g.copy()
print(f'\n||g_pin - T^T g_mj|| = {np.linalg.norm(g_pin - T.T @ g_mj):.3e}')
print(f'||g_pin||            = {np.linalg.norm(g_pin):.3e}')

## 4. 중심 운동량 — linear/angular momentum 직접 비교 ✅

MuJoCo는 `data.subtree_linvel`, `data.subtree_angmom`로 직접 제공 (단, `mj_subtreeVel` 호출 필요).

In [ ]:
mujoco.mj_subtreeVel(kernel.model, kernel.data)

lin_pin = np.asarray(wrapper.hg.linear)
ang_pin = np.asarray(wrapper.hg.angular)
lin_mj  = kernel.data.subtree_linvel[1] * wrapper.mass     # m·v_com
ang_mj  = kernel.data.subtree_angmom[1].copy()              # angular momentum about CoM

print(f'[lin_mom] pin = {lin_pin}')
print(f'          mj  = {lin_mj}')
print(f'          err = {np.linalg.norm(lin_pin - lin_mj):.3e}')

print(f'\n[ang_mom] pin = {ang_pin}')
print(f'          mj  = {ang_mj}')
print(f'          err = {np.linalg.norm(ang_pin - ang_mj):.3e}')

### 4.1 Centroidal Composite Inertia `Ig` — 수동 합성

MuJoCo는 직접 제공 X. 각 body의 spatial inertia를 CoM frame world-aligned로 옮겨 합산.

In [ ]:
def skew(v):
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])

def composite_inertia_about_com_mj(model, data, com_world):
    """각 body inertia를 com_world 기준 world-aligned로 옮겨 합산."""
    Ig = np.zeros((3, 3))
    for i in range(1, model.nbody):
        m_i = model.body_mass[i]
        if m_i == 0: continue
        I_local = np.diag(model.body_inertia[i])             # principal inertia diag
        # MuJoCo iquat: body principal axes → body frame  (w,x,y,z)
        R_p2b = np.zeros(9)
        mujoco.mju_quat2Mat(R_p2b, model.body_iquat[i])
        R_p2b = R_p2b.reshape(3, 3)
        R_b2w = data.xmat[i].reshape(3, 3)
        R = R_b2w @ R_p2b
        I_world = R @ I_local @ R.T
        # parallel axis: body com world = xipos[i]
        r = data.xipos[i] - com_world
        S = skew(r)
        Ig += I_world - m_i * (S @ S)
    return Ig

Ig_mj  = composite_inertia_about_com_mj(kernel.model, kernel.data, wrapper.com_pos_world)
Ig_pin = wrapper.inertia_matrix

print('Ig pin:\n', Ig_pin)
print('\nIg mj :\n', Ig_mj)
print(f'\n||Ig_pin - Ig_mj||_F = {np.linalg.norm(Ig_pin - Ig_mj):.3e}')
print(f'||Ig_pin||_F          = {np.linalg.norm(Ig_pin):.3e}')
print(f'relative err = {np.linalg.norm(Ig_pin - Ig_mj)/np.linalg.norm(Ig_pin)*100:.2f}%')

## 5. Jdot @ dq (frame acceleration drift)

MuJoCo는 `mj_jacDot`로 직접 제공 (≥3.0).

In [ ]:
for key in ['L_foot', 'R_foot']:
    body_id = mujoco.mj_name2id(kernel.model, mujoco.mjtObj.mjOBJ_BODY, PIN_TO_MJBODY[key])
    R_body  = kernel.data.xmat[body_id].reshape(3, 3)
    p_world = kernel.data.xpos[body_id] + R_body @ FOOT_SOLE_OFFSET
    
    Jdotp = np.zeros((3, nv))
    Jdotr = np.zeros((3, nv))
    mujoco.mj_jacDot(kernel.model, kernel.data, Jdotp, Jdotr, p_world, body_id)
    Jdot_mj  = np.vstack([Jdotp, Jdotr])
    Jd_dq_mj = Jdot_mj @ kernel.data.qvel
    
    Jd_dq_pin = wrapper.get_Jdot_dq_world(key)
    err = np.linalg.norm(Jd_dq_pin - Jd_dq_mj)
    print(f'[{key}] Jdot·dq')
    print(f'       pin = {Jd_dq_pin}')
    print(f'       mj  = {Jd_dq_mj}')
    print(f'       err = {err:.3e}\n')

## 6. Coriolis matrix C — MuJoCo는 직접 제공 X

MuJoCo는 `qfrc_bias = C·dq + g` 형태로만 제공. C 단독이 필요하면 RNEA 호출이나 finite difference로 추출. 일반 control에선 `qfrc_bias`만으로 충분.

In [ ]:
C_pin = wrapper.C
Cdq_g = C_pin @ state.dq + wrapper.g
print(f'[Pin self-check] ||C·dq + g - nle|| = {np.linalg.norm(Cdq_g - wrapper.nle):.3e}')
print('\nMuJoCo: C 행렬 단독 access X. qfrc_bias = C·dq + g 형태로만 제공.')

## 7. Centroidal Momentum Matrix `Ag` — MuJoCo는 직접 제공 X (수동 합성 필요)

Pinocchio의 `data.Ag`는 `hg = Ag·dq`를 만족하는 6×nv 행렬. MuJoCo는 이를 직접 주지 않으므로, 각 body의 spatial inertia + Jacobian을 CoM frame으로 합성해야 한다.

그러나 momentum 자체(hg)는 위에서 보았듯이 `subtree_linvel`/`subtree_angmom`로 직접 얻을 수 있어, **Ag가 필요한 경우는 centroidal MPC처럼 행렬 자체가 cost/constraint에 들어갈 때 뿐**이다.

In [ ]:
# 주의: cell 17의 wrapper.C 호출이 data.Ag를 덮어씀 → 다시 update_model 호출
wrapper.update_model(state.q, state.dq)

# 검증: Ag_pin @ dq_pin == hg_pin (Pin self-check)
Ag_pin = wrapper.Ag.copy()
hg_pred = Ag_pin @ state.dq
hg_true = np.concatenate([wrapper.hg.linear, wrapper.hg.angular])
print(f'[Pin self-check] ||Ag·dq - hg|| = {np.linalg.norm(hg_pred - hg_true):.3e}')

# MJ 우회: subtree로 hg 직접 (Ag 행렬 없이도 momentum 자체는 OK)
mujoco.mj_subtreeVel(kernel.model, kernel.data)
lin_mj_ = kernel.data.subtree_linvel[1] * wrapper.mass
ang_mj_ = kernel.data.subtree_angmom[1]
print(f'\nMJ direct momentum (no Ag needed):')
print(f'  lin = {lin_mj_}')
print(f'  ang = {ang_mj_}')
print(f'  err vs Pin hg: {np.linalg.norm(np.r_[lin_mj_, ang_mj_] - hg_true):.3e}')


# ───────────────────────────────────────────────────────────────────
# Ag matrix 자체를 MuJoCo로 합성 (centroidal MPC / WBC null-space에서 필요)
# Ag = sum_i (body i의 spatial momentum 기여를 world CoM 기준 frame으로 옮긴 것)
#   Ag[0:3,:] = Σ m_i · J_p,i                              (linear)
#   Ag[3:6,:] = Σ ( I_i_world · J_r,i + [r_i]× · m_i · J_p,i )   (angular about CoM)
# 여기서 J_p,i, J_r,i 는 "body i의 CoM(=xipos[i])에서의" 6×nv Jacobian
# ───────────────────────────────────────────────────────────────────
def compute_Ag_mj(model, data):
    nv = model.nv
    com = data.subtree_com[1]   # whole-body CoM (pelvis subtree)
    Ag = np.zeros((6, nv))
    for i in range(1, model.nbody):
        m_i = model.body_mass[i]
        if m_i == 0: continue

        # body i CoM world 위치
        ci = data.xipos[i]
        # body i CoM에서의 6×nv Jacobian (LOCAL_WORLD_ALIGNED 등가)
        jacp = np.zeros((3, nv))
        jacr = np.zeros((3, nv))
        mujoco.mj_jac(model, data, jacp, jacr, ci, i)

        # body i의 world-aligned inertia tensor (about body CoM)
        I_local = np.diag(model.body_inertia[i])             # principal axes diag
        R_p2b = np.zeros(9)
        mujoco.mju_quat2Mat(R_p2b, model.body_iquat[i])
        R_p2b = R_p2b.reshape(3, 3)
        R_b2w = data.xmat[i].reshape(3, 3)
        R = R_b2w @ R_p2b
        Ii_world = R @ I_local @ R.T

        # body CoM → whole-body CoM 으로 옮기기
        r = ci - com
        S = np.array([[0, -r[2], r[1]], [r[2], 0, -r[0]], [-r[1], r[0], 0]])

        Ag[0:3, :] += m_i * jacp
        Ag[3:6, :] += Ii_world @ jacr + S @ (m_i * jacp)
    return Ag


# 1) Ag_mj @ qvel == hg (subtree)?
Ag_mj = compute_Ag_mj(kernel.model, kernel.data)
hg_from_Ag = Ag_mj @ kernel.data.qvel
hg_subtree = np.r_[kernel.data.subtree_linvel[1] * wrapper.mass,
                   kernel.data.subtree_angmom[1]]
print(f'\n[Ag_mj @ qvel == subtree hg] err = {np.linalg.norm(hg_from_Ag - hg_subtree):.3e}')

# 2) Pin Ag와 행렬째 비교 — 좌표 변환: Ag_pin @ dq_pin == Ag_mj @ qvel_mj, qvel_mj = T·dq_pin
#    →  Ag_pin = Ag_mj @ T
Ag_mj_to_pin = Ag_mj @ T
err_mat = np.linalg.norm(Ag_pin - Ag_mj_to_pin)
print(f'[Ag_pin - Ag_mj @ T]_F     = {err_mat:.3e}  (||Ag_pin||={np.linalg.norm(Ag_pin):.2e})')
print(f'  relative err = {err_mat / np.linalg.norm(Ag_pin) * 100:.3f}%')

## 종합 비교표 (실측치)

| Pinocchio_Wrapper 양 | MuJoCo 등가 | 일치도 (이 노트북) | 비고 |
|---|---|---|---|
| `M` | `mj_fullM(qM)` + `T^T·T` | **0.09%** (rel) | 모델 inertia 미세 차이 |
| `nle` | `data.qfrc_bias` + `T^T` | ~0.04% (rel) | 동상 |
| `g` | `qfrc_bias` at `qvel=0` + `T^T` | ~0.01% (rel) | |
| `M_inv` | `np.linalg.inv(M_mj)` | (M과 동일) | |
| `C` matrix | **직접 X** | — | 필요시 RNEA / finite-diff |
| `base_pos` | `data.xpos[body_id]` | **0** | OK |
| `com_pos_world` | `data.subtree_com[1]` | **6e-6** | OK |
| `com_vel_world` | `data.subtree_linvel[1]` | **~1e-5** | OK |
| `J_com` | `mj_jacSubtreeCom(.., 1)` + `·T` | **2e-5** | OK |
| Frame Jacobian (foot sole) | `mj_jac(p, body_id)` + `·T` | **2e-8** | OK |
| Frame placement | `data.xpos`/`data.xmat` (+offset) | **5e-9** | OK |
| Frame velocity | `J_world @ qvel` | **1e-9** | OK |
| `Jdot @ dq` | `mj_jacDot` | **3e-3** (주의) | offset point 처리 차이 가능 |
| linear momentum (`hg.linear`) | `m · subtree_linvel[1]` | **6e-5** | OK |
| angular momentum (`hg.angular`) | `subtree_angmom[1]` | **4e-6** | OK |
| centroidal inertia `Ig` (3x3) | parallel-axis 수동 합성 | **2e-5** | OK |
| **centroidal matrix `Ag` (6xnv)** | **직접 X**, 수동 합성 필요 | — | 어려움 |

## 핵심 결론

1. **kinematics 양은 MuJoCo로 1e-5 이하 오차로 완벽 재현.** 두 가지 변환만 기억:
   - quat 순서: MJ `(w,x,y,z)` ↔ Pin `(x,y,z,w)`
   - free joint linear vel 좌표계: `T = blkdiag(R_body_to_world, I, I)`로 보정

2. **dynamics 양(M, nle, g)도 변환은 정확**하지만 URDF/MJCF의 inertia 정의 차이로 ~0.1% 잔차. 같은 원본 모델이라도 두 export 파이프라인이 inertia를 약간 다르게 계산하는 게 일반적. 정확한 일치를 원하면 **MJCF의 inertia로 URDF를 재생성**하거나 그 반대.

3. **Centroidal Inertia `Ig`도 직접 합성으로 거의 완벽 일치** (2e-5). MuJoCo만으로도 충분.

4. **MuJoCo가 진짜 약한 두 가지:**
   - `Coriolis matrix C` 단독 (qfrc_bias로만 제공)
   - `Centroidal Momentum Matrix Ag` (6xnv 자체는 수동 합성)

   둘 다 *행렬 자체가 cost/constraint에 들어가는 알고리즘*에서만 issue. 보통은 우회 가능.

5. **Frame 정의 차이 주의**: Pinocchio의 가상 OP_FRAME(예: `L_foot_sole`)이 MJCF엔 없음. `<site>` 추가하거나 body offset으로 직접 계산.

→ **`Pinocchio_Wrapper`의 거의 모든 기능은 순수 MuJoCo로 직접 구현 가능.** centroidal dynamics가 핵심인 알고리즘에서만 Pinocchio가 압도적으로 편하다.
